In [14]:
# ============================================================
# End-to-End Deep Network Generalization Test
#
# PURPOSE: Test whether the scaling law and epoch-time illusion
# survive beyond frozen-feature linear probes, on real images
# with an end-to-end fine-tuned ResNet-50.
#
# DATASET: Waterbirds canonical (Sagawa et al. 2020)
#   - Standard Group DRO benchmark
#   - 4 groups: {landbird,waterbird} x {land,water background}
#   - Known collapse: waterbird-on-land group suppressed
#   - Already downloaded in Notebook 3
#
# DESIGN:
#   Three sweeps, each comparing EPOCH vs STEP collapse time:
#
#   Sweep A — Clock artifact test (the make-or-break experiment)
#     Fix ratio~2.2, p~0.23, vary absolute N by subsampling
#     If step-space N exponent ≈ 0: epoch-time illusion survives
#     If step-space N exponent < 0: illusion is probe-specific
#
#   Sweep B — p exponent (minority-class fraction)
#     Fix N, ratio, vary p by subsampling minority class
#     Expect: collapse time ~ p^{0.5} (derived result)
#
#   Sweep C — R exponent (size ratio)
#     Fix N_maj, p, vary ratio by subsampling minority pool
#     Expect: collapse time ~ R^{-0.43}
#
# REGIME MANAGEMENT (critical — see note):
#   Fine-tuning with default settings hits overparameterized
#   regime where DRO ≡ ERM (Sagawa et al. finding).
#   Must use strong weight decay (lambda=1.0) or early stopping
#   to stay in the non-vanishing-loss regime where q-dynamics
#   are active. We use both and report which fires.
#
# KEY QUESTION:
#   Do the SIGNS of all exponents survive? Do the magnitudes
#   stay in the right ballpark? Does the clock artifact survive?
#   Any of those failing requires re-scoping the paper.
#
# ARCHITECTURE: ResNet-50, ImageNet pretrained
# OPTIMIZER: SGD with momentum (Sagawa et al. protocol)
# DRO step size: eta=0.01 (smaller than synthetic — avoids EoS
#   with real image gradients whose scale differs from Gaussians)
# SEEDS: 3 (T4 runtime limit)
#
# RUNTIME: ~2.5h on Kaggle T4
# DATASET: /kaggle/input/waterbird (already mounted from Nb3)
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from PIL import Image
import os, json, math, warnings
warnings.filterwarnings('ignore')

# ── Constants ─────────────────────────────────────────────────
SEEDS         = [42, 0, 1]
MAX_EPOCHS    = 30         # enough to see collapse before overfitting
BATCH_SIZE    = 32
SGD_LR        = 1e-3       # Sagawa et al. protocol for Waterbirds
SGD_MOMENTUM  = 0.9
WEIGHT_DECAY  = 1.0        # strong WD to stay out of overparameterized regime
DRO_ETA       = 0.01       # smaller than synthetic; real image gradients differ
COLLAPSE_THR  = 0.01
N_GROUPS      = 4          # {landbird,waterbird} x {land,water}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Weight decay: {WEIGHT_DECAY}  (strong, per Sagawa et al.)')
print(f'DRO eta: {DRO_ETA}')
print(f'Regime: non-vanishing-loss (WD prevents zero training loss)')
print(f'Sweeps: A=clock artifact  B=p exponent  C=R exponent')

Device: cuda
Weight decay: 1.0  (strong, per Sagawa et al.)
DRO eta: 0.01
Regime: non-vanishing-loss (WD prevents zero training loss)
Sweeps: A=clock artifact  B=p exponent  C=R exponent


In [15]:
# ── CELL 2: Dataset loader ─────────────────────────────────────
class WaterbirdsDataset(Dataset):
    """
    Loads Waterbirds from the canonical Sagawa et al. metadata.csv.
    group = y*2 + place  (0..3)
    Returns (image_tensor, label, group).
    """
    def __init__(self, metadata_path, img_root, split='train',
                 transform=None):
        df = pd.read_csv(metadata_path)
        # split: 0=train 1=val 2=test
        split_map = {'train': 0, 'val': 1, 'test': 2}
        df = df[df['split'] == split_map[split]].reset_index(drop=True)
        self.df        = df
        self.img_root  = img_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.img_root, row['img_filename'])
        img   = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row['y'])
        place = int(row['place'])
        group = label * 2 + place
        return img, label, group


# Transforms
TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Find metadata and image root
META_PATH = None
IMG_ROOT  = None
for search in ['/kaggle/working/waterbirds_canonical',
               '/kaggle/input/waterbird',
               '/kaggle/input/waterbirds']:
    for root, dirs, files in os.walk(search):
        if 'metadata.csv' in files:
            candidate = os.path.join(root, 'metadata.csv')
            df_check  = pd.read_csv(candidate, nrows=3)
            if 'y' in df_check.columns and 'place' in df_check.columns:
                META_PATH = candidate
                IMG_ROOT  = root
                break
    if META_PATH:
        break

if META_PATH is None:
    raise FileNotFoundError(
        'Waterbirds metadata.csv not found. '
        'Mount the Waterbirds dataset or run Notebook 3 first.')

print(f'Metadata: {META_PATH}')
print(f'Image root: {IMG_ROOT}')

# Load full training set
full_train = WaterbirdsDataset(META_PATH, IMG_ROOT, 'train',
                               TRAIN_TRANSFORM)
full_test  = WaterbirdsDataset(META_PATH, IMG_ROOT, 'test',
                               EVAL_TRANSFORM)

df_meta    = full_train.df
print(f'\nFull train size: {len(full_train)}')
print('Group counts:')
for g in range(4):
    lbl = g // 2; plc = g % 2
    cnt = ((df_meta['y']==lbl)&(df_meta['place']==plc)).sum()
    name = f'{"water" if lbl else "land"}bird+{"water" if plc else "land"}'
    print(f'  g={g} ({name}): {cnt}')
print(f'\nTest size: {len(full_test)}')

Metadata: /kaggle/working/waterbirds_canonical/metadata.csv
Image root: /kaggle/working/waterbirds_canonical

Full train size: 4795
Group counts:
  g=0 (landbird+land): 3498
  g=1 (landbird+water): 184
  g=2 (waterbird+land): 56
  g=3 (waterbird+water): 1057

Test size: 5794


In [16]:
# ── CELL 3: ResNet-50 + DRO runner ────────────────────────────
def get_model():
    model = models.resnet50(pretrained=True)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model.to(device)


def evaluate_groups(model, loader):
    model.eval()
    preds_all, labels_all, groups_all = [], [], []
    with torch.no_grad():
        for xb, yb, gb in loader:
            logits = model(xb.to(device)).cpu()
            preds_all.append(logits.argmax(1))
            labels_all.append(yb)
            groups_all.append(gb)
    preds  = torch.cat(preds_all).numpy()
    labels = torch.cat(labels_all).numpy()
    groups = torch.cat(groups_all).numpy()
    acc = {}
    for g in range(N_GROUPS):
        mask = groups == g
        if mask.sum() > 0:
            acc[g] = float((preds[mask]==labels[mask]).mean())
    avg_acc = float((preds==labels).mean())
    return acc, avg_acc


def run_dro_resnet(train_dataset, test_dataset,
                   eta=DRO_ETA, seed=42,
                   max_epochs=MAX_EPOCHS,
                   weight_decay=WEIGHT_DECAY):
    """
    End-to-end fine-tuning with Group DRO.
    Records collapse_epoch, collapse_step, per-group accuracy traces.
    Returns dict with timescale results.
    """
    torch.manual_seed(seed); np.random.seed(seed)

    tr_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=2, pin_memory=True)
    te_loader = DataLoader(test_dataset, batch_size=64,
                           shuffle=False, num_workers=2, pin_memory=True)

    model  = get_model()
    opt    = optim.SGD(model.parameters(), lr=SGD_LR,
                       momentum=SGD_MOMENTUM, weight_decay=weight_decay)
    sched  = optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.1)

    q = torch.ones(N_GROUPS).to(device) / N_GROUPS

    # Track which groups appear in this training subset
    groups_in_train = set()
    for _, _, gb in tr_loader:
        groups_in_train.update(gb.numpy().tolist())
        break  # one batch sufficient

    steps_per_epoch = math.ceil(len(train_dataset) / BATCH_SIZE)
    collapse_epoch  = None
    collapse_step   = None
    global_step     = 0

    epoch_records = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb, gb in tr_loader:
            xb, yb, gb = xb.to(device), yb.to(device), gb.to(device)
            opt.zero_grad()
            logits = model(xb)
            losses = nn.CrossEntropyLoss(reduction='none')(logits, yb)

            g_loss = torch.zeros(N_GROUPS).to(device)
            for g in range(N_GROUPS):
                mask = gb == g
                if mask.sum() > 0:
                    g_loss[g] = losses[mask].mean()

            q = q * torch.exp(eta * g_loss.detach())
            q = q / q.sum()
            (q * g_loss).sum().backward()
            opt.step()
            global_step += 1

            min_q = float(q.min().item())
            if collapse_step is None and min_q < COLLAPSE_THR:
                collapse_step  = global_step
                collapse_epoch = epoch

        sched.step()

        # Evaluate
        group_acc, avg_acc = evaluate_groups(model, te_loader)
        min_q_val = float(q.min().item())
        epoch_records.append({
            'epoch':    epoch,
            'step':     global_step,
            'min_q':    round(min_q_val, 5),
            'avg_acc':  round(avg_acc, 4),
            'group_acc': {str(g): round(v, 4)
                          for g, v in group_acc.items()},
        })

        if epoch % 5 == 0:
            worst_acc = min(group_acc.values()) if group_acc else 0
            print(f'    Epoch {epoch:2d}: min_q={min_q_val:.4f}  '
                  f'avg_acc={avg_acc:.3f}  worst_acc={worst_acc:.3f}  '
                  f'step={global_step}')

    final_group_acc, final_avg_acc = evaluate_groups(model, te_loader)
    del model
    torch.cuda.empty_cache()

    return {
        'collapse_epoch': collapse_epoch if collapse_epoch else max_epochs + 1,
        'collapse_step':  collapse_step  if collapse_step  else global_step + 1,
        'collapsed':      collapse_epoch is not None,
        'steps_per_epoch': steps_per_epoch,
        'final_avg_acc':  round(final_avg_acc, 4),
        'final_group_acc': {str(g): round(v, 4)
                            for g, v in final_group_acc.items()},
        'epoch_records':  epoch_records,
    }


print('ResNet-50 DRO runner defined.')
print(f'Steps per epoch (full train ~{len(full_train)} imgs): '
      f'~{math.ceil(len(full_train)/BATCH_SIZE)}')

ResNet-50 DRO runner defined.
Steps per epoch (full train ~4795 imgs): ~150


In [17]:
# ── CELL 4: Subsampling utilities ─────────────────────────────
def subsample_by_group(dataset, target_group_counts, seed=42):
    """
    Returns a Subset with target_group_counts[g] examples per group.
    target_group_counts: dict {group_id: n_target}
    Missing groups or n_target > available → use all available.
    """
    rng  = np.random.RandomState(seed)
    df   = dataset.df
    idxs = []
    for g, n_target in target_group_counts.items():
        lbl  = g // 2; plc = g % 2
        pool = df[(df['y']==lbl)&(df['place']==plc)].index.tolist()
        n    = min(n_target, len(pool))
        sel  = rng.choice(pool, n, replace=False).tolist()
        # Convert df index to positional index
        pos  = [df.index.get_loc(i) for i in sel]
        idxs.extend(pos)
    return Subset(dataset, idxs)


def subsample_minority_class(dataset, p_target, seed=42):
    """
    Vary minority-class fraction p within the minority demographic.
    Minority demographic = waterbird (y=1).
    Minority class within it = waterbird-on-land (g=3).
    Majority class within it = waterbird-on-water (g=2).
    Subsample g=3 to achieve target p.
    """
    rng = np.random.RandomState(seed)
    df  = dataset.df

    # Majority group stays full
    maj_idxs = df[(df['y']==0)].index.tolist()
    ww_idxs  = df[(df['y']==1)&(df['place']==1)].index.tolist()
    wl_idxs  = df[(df['y']==1)&(df['place']==0)].index.tolist()

    # n_wl = p * (n_wl + n_ww) => n_wl = p/(1-p) * n_ww
    n_ww   = len(ww_idxs)
    n_wl_t = int(p_target / (1 - p_target) * n_ww)
    n_wl_t = min(n_wl_t, len(wl_idxs))
    n_wl_t = max(1, n_wl_t)

    wl_sel = rng.choice(wl_idxs, n_wl_t, replace=False).tolist()
    all_idxs = maj_idxs + ww_idxs + wl_sel
    pos = [df.index.get_loc(i) for i in all_idxs]
    actual_p = n_wl_t / (n_wl_t + n_ww)
    return Subset(dataset, pos), actual_p


# Print group counts in full dataset
print('Full Waterbirds train group counts:')
df_meta_t = full_train.df
for g in range(4):
    lbl = g // 2; plc = g % 2
    cnt = ((df_meta_t['y']==lbl)&(df_meta_t['place']==plc)).sum()
    print(f'  g={g}: {cnt}')

# Natural ratio and p
n_g2 = ((df_meta_t['y']==1)&(df_meta_t['place']==1)).sum()
n_g3 = ((df_meta_t['y']==1)&(df_meta_t['place']==0)).sum()
n_g0 = ((df_meta_t['y']==0)&(df_meta_t['place']==0)).sum()
n_g1 = ((df_meta_t['y']==0)&(df_meta_t['place']==1)).sum()
N_MIN = n_g2 + n_g3   # minority demographic (waterbird)
N_MAJ = n_g0 + n_g1   # majority demographic (landbird)
P_NAT = n_g3 / (n_g2 + n_g3)  # natural p
R_NAT = N_MAJ / N_MIN

print(f'\nNatural R = {R_NAT:.2f}  p = {P_NAT:.3f}')
print(f'N_min={N_MIN}  N_maj={N_MAJ}')

Full Waterbirds train group counts:
  g=0: 3498
  g=1: 184
  g=2: 56
  g=3: 1057

Natural R = 3.31  p = 0.050
N_min=1113  N_maj=3682


In [18]:
# ── CELL 5: SWEEP A — Clock artifact test ─────────────────────
# Fix ratio≈natural, vary absolute scale by subsampling both groups
# proportionally. If clock artifact holds: step exponent ≈ 0.
print('='*60)
print('SWEEP A: Clock artifact test')
print(f'Fix ratio≈{R_NAT:.1f}, vary absolute scale')
print('Prediction: collapse_step FLAT; collapse_epoch falls')
print('='*60)

# Scale factors: 1.0=full, 0.5=half, 0.25=quarter, 0.125=eighth
SCALE_FACTORS = [0.125, 0.25, 0.5, 1.0]

records_A = []
for scale in SCALE_FACTORS:
    # Subsample both demographics proportionally
    n_min_t = max(10, int(N_MIN * scale))
    n_maj_t = max(10, int(N_MAJ * scale))
    # Keep ratio fixed by computing per-group counts
    n_g0_t = max(1, int(n_g0 * scale))
    n_g1_t = max(1, int(n_g1 * scale))
    n_g2_t = max(1, int(n_g2 * scale))
    n_g3_t = max(1, int(n_g3 * scale))
    actual_N_min = n_g2_t + n_g3_t
    actual_N_maj = n_g0_t + n_g1_t
    actual_R     = actual_N_maj / actual_N_min

    epoch_list, step_list, spe_list = [], [], []
    print(f'\n  scale={scale:.3f}  N_min={actual_N_min}  '
          f'N_maj={actual_N_maj}  R={actual_R:.2f}')

    for seed in SEEDS:
        sub = subsample_by_group(
            full_train,
            {0: n_g0_t, 1: n_g1_t, 2: n_g2_t, 3: n_g3_t},
            seed=seed)
        res = run_dro_resnet(sub, full_test, seed=seed)
        epoch_list.append(res['collapse_epoch'])
        step_list.append(res['collapse_step'])
        spe_list.append(res['steps_per_epoch'])
        print(f'    seed {seed}: epoch={res["collapse_epoch"]}  '
              f'step={res["collapse_step"]}  '
              f'collapsed={res["collapsed"]}  '
              f'avg_acc={res["final_avg_acc"]:.3f}')

    mean_ep  = float(np.mean(epoch_list))
    mean_st  = float(np.mean(step_list))
    mean_spe = float(np.mean(spe_list))
    col_rate = sum(e <= MAX_EPOCHS for e in epoch_list) / len(SEEDS)
    censored = sum(e > MAX_EPOCHS for e in epoch_list)

    records_A.append({
        'scale': scale,
        'N_min': actual_N_min, 'N_maj': actual_N_maj,
        'ratio': round(actual_R, 2),
        'mean_epoch': round(mean_ep, 2), 'mean_step': round(mean_st, 1),
        'mean_spe':   round(mean_spe, 1),
        'collapse_rate': round(col_rate, 3), 'censored': censored,
        'log_N_min': round(math.log(actual_N_min), 4),
        'log_epoch': round(math.log(mean_ep), 4) if mean_ep > 0 else None,
        'log_step':  round(math.log(mean_st), 4) if mean_st > 0 else None,
    })

df_A = pd.DataFrame(records_A)
print('\n=== SWEEP A RESULTS ===')
print(df_A[['scale','N_min','mean_epoch','mean_step','mean_spe',
            'collapse_rate']].to_string())

# Fit exponents
fit_A = df_A[(df_A['censored']==0) & df_A['log_step'].notna()]
if len(fit_A) >= 2:
    sl_ep, _, r_ep, _, _ = stats.linregress(
        fit_A['log_N_min'], fit_A['log_epoch'])
    sl_st, _, r_st, _, _ = stats.linregress(
        fit_A['log_N_min'], fit_A['log_step'])
    print(f'\nEpoch fit: epoch ~ N_min^{sl_ep:.3f}  (R²={r_ep**2:.3f})')
    print(f'Step  fit: step  ~ N_min^{sl_st:.3f}  (R²={r_st**2:.3f})')
    print()
    if abs(sl_st) < 0.2:
        print('=> CLOCK ARTIFACT SURVIVES end-to-end: step exponent ≈ 0')
        print('   The epoch-time illusion generalizes beyond frozen probes.')
    elif sl_st < -0.2:
        print(f'=> STEP EXPONENT {sl_st:.2f} < 0: size is partially real end-to-end')
        print('   The illusion is partly probe-specific. Re-scope paper.')
    else:
        print(f'=> AMBIGUOUS: step exponent {sl_st:.2f} with {len(fit_A)} clean points')

SWEEP A: Clock artifact test
Fix ratio≈3.3, vary absolute scale
Prediction: collapse_step FLAT; collapse_epoch falls

  scale=0.125  N_min=139  N_maj=460  R=3.31
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 207MB/s] 


    Epoch  5: min_q=0.2094  avg_acc=0.778  worst_acc=0.000  step=85
    Epoch 10: min_q=0.1728  avg_acc=0.778  worst_acc=0.000  step=170
    Epoch 15: min_q=0.1253  avg_acc=0.767  worst_acc=0.121  step=255
    Epoch 20: min_q=0.0918  avg_acc=0.772  worst_acc=0.154  step=340
    Epoch 25: min_q=0.0684  avg_acc=0.750  worst_acc=0.204  step=425
    Epoch 30: min_q=0.0495  avg_acc=0.735  worst_acc=0.206  step=510
    seed 42: epoch=31  step=511  collapsed=False  avg_acc=0.735
    Epoch  5: min_q=0.2193  avg_acc=0.778  worst_acc=0.000  step=85
    Epoch 10: min_q=0.1850  avg_acc=0.778  worst_acc=0.000  step=170
    Epoch 15: min_q=0.1441  avg_acc=0.780  worst_acc=0.293  step=255
    Epoch 20: min_q=0.1118  avg_acc=0.767  worst_acc=0.260  step=340
    Epoch 25: min_q=0.0837  avg_acc=0.770  worst_acc=0.248  step=425
    Epoch 30: min_q=0.0618  avg_acc=0.764  worst_acc=0.262  step=510
    seed 0: epoch=31  step=511  collapsed=False  avg_acc=0.764
    Epoch  5: min_q=0.2120  avg_acc=0.778  wors